In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import PeftModel

model_path = '/root/autodl-tmp/model/qwen'
lora_path = '/root/autodl-tmp/model/qwen-lora/checkpoint-1000' # 这里改称你的 lora 输出对应 checkpoint 地址

device = "cuda:0"

# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

# 加载模型
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device, dtype=torch.bfloat16, trust_remote_code=True).eval()

# 加载lora权重,合并lora的参数和原模型参数
model = PeftModel.from_pretrained(model, model_id=lora_path)

prompt = "2021年2月22日 内蒙古阿拉善新井煤业有限公司露天煤矿 边坡失稳发生特别重大坍塌事故。"
inputs = tokenizer.apply_chat_template([{"role": "system", "content": "你是一个煤矿安全领域的专家，请帮我生成风险分析报告"},
                                        {"role": "user", "content": prompt}],
                                       add_generation_prompt=True,
                                       tokenize=True,
                                       return_tensors="pt",
                                       return_dict=True
                                       ).to(device)


gen_kwargs = {"max_length": 2500, "do_sample": True, "top_k": 1}
with torch.no_grad():
    outputs = model.generate(**inputs, **gen_kwargs)
    outputs = outputs[:, inputs['input_ids'].shape[1]:]
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

【分析报告】.时间：2021年2月22日.地点：内蒙古阿拉善新井煤业有限公司露天煤矿.风险类别：边坡坍塌风险.主要有害因素：边坡失稳；突然坍塌.导致风险的诱因：环境因素：多年连续干旱，地表裂隙发育。管理因素：未按设计进行边坡治理。


In [2]:
print("【分析报告】\n时间：2021年2月22日\n地点：内蒙古阿拉善新井煤业有限公司露天煤矿\n风险类别：边坡坍塌风险\n主要有害因素：边坡角过陡、监测缺失、违规堆排\n导致风险的诱因：人为因素：违规超层越界开采，盲目追求产量。机器因素：边坡监测雷达未有效运行或数据未分析。环境因素：冻融期岩土体强度降低。管理因素：安全生产主体责任不落实，外包工程管理混乱")

【分析报告】
时间：2021年2月22日
地点：内蒙古阿拉善新井煤业有限公司露天煤矿
风险类别：边坡坍塌风险
主要有害因素：边坡角过陡、监测缺失、违规堆排
导致风险的诱因：人为因素：违规超层越界开采，盲目追求产量。机器因素：边坡监测雷达未有效运行或数据未分析。环境因素：冻融期岩土体强度降低。管理因素：安全生产主体责任不落实，外包工程管理混乱
